In [ ]:
# Modulo de llamadas http para descargar ficheros
!pip install requests

# Libreria del problema TSP: http://elib.zib.de/pub/mp-testdata/tsp/tsplib/tsplib.html
!pip install tsplib95

In [ ]:
import tsplib95
import random
from math import e
import urllib.request
import os
import gzip
import shutil

In [ ]:
# DATOS DEL PROBLEMA
file = "swiss42.tsp"
if not os.path.exists(file):
    urllib.request.urlretrieve("http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/swiss42.tsp.gz", file + '.gz')
    with gzip.open(file + '.gz', 'rb') as f_in:
        with open(file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    os.remove(file + '.gz')
    print('Fichero descargado y descomprimido')
else:
    print('Fichero ya disponible localmente')

problem = tsplib95.load(file)

# Nodos
Nodos = list(problem.get_nodes())

# Devuelve la distancia entre dos nodos
def distancia(a, b, problem):
    return problem.get_weight(a, b)

# Devuelve la distancia total de una trayectoria/solucion(lista de nodos)
def distancia_total(solucion, problem):
    distancia_total = 0
    for i in range(len(solucion)-1):
        distancia_total += distancia(solucion[i], solucion[i+1], problem)
    return distancia_total + distancia(solucion[len(solucion)-1], solucion[0], problem)


##Algoritmo de colonia de hormigas

La función Add_Nodo selecciona al azar un nodo con probabilidad uniforme.
Para ser mas eficiente debería seleccionar el próximo nodo siguiendo la probabilidad correspondiente a la ecuación:

$p^k_{ij}(t) = \frac{[\tau_{ij}(t)]^\alpha[\nu_{ij}]^\beta}{\sum_{l\in J^k_i} [\tau_{il}(t)]^\alpha[\nu_{il}]^\beta}$, si $j \in J^k_i$

$p^k_{ij}(t) = 0$, si $j \notin J^k_i$

In [ ]:
# Parametros del algoritmo ACO
ALPHA = 1    # Importancia de la feromona
BETA = 2     # Importancia de la visibilidad (inverso de la distancia)
RHO = 0.1    # Tasa de evaporacion de feromonas

def Add_Nodo(problem, H, T):
    """
    Selecciona el siguiente nodo usando la ecuacion de probabilidad ACO:
    p_ij = (tau_ij^alpha * nu_ij^beta) / sum(tau_il^alpha * nu_il^beta)
    donde tau = feromona, nu = 1/distancia (visibilidad)
    """
    Nodos = list(problem.get_nodes())
    nodo_actual = H[-1]
    candidatos = list(set(range(0, len(Nodos))) - set(H))

    if not candidatos:
        return H[0]  # Volver al inicio si no hay candidatos

    # Calcular probabilidades para cada nodo candidato
    probabilidades = []
    for j in candidatos:
        tau = T[nodo_actual][j]                                    # Feromona en la arista (i,j)
        dist = distancia(nodo_actual, j, problem)
        nu = 1.0 / max(dist, 0.001)                               # Visibilidad = 1/distancia
        probabilidades.append((tau ** ALPHA) * (nu ** BETA))

    # Normalizar probabilidades
    suma_total = sum(probabilidades)
    if suma_total == 0:
        probabilidades = [1.0 / len(candidatos)] * len(candidatos)
    else:
        probabilidades = [p / suma_total for p in probabilidades]

    # Seleccion por ruleta
    r = random.random()
    acumulado = 0
    for idx, prob in enumerate(probabilidades):
        acumulado += prob
        if r <= acumulado:
            return candidatos[idx]

    return candidatos[-1]


def Incrementa_Feromona(problem, T, H):
    """Incrementa segun la calidad de la solucion. Inversamente proporcional a la distancia total."""
    Q = 1000  # Constante de deposito de feromonas
    deposito = Q / distancia_total(H, problem)
    for i in range(len(H)-1):
        T[H[i]][H[i+1]] += deposito
        T[H[i+1]][H[i]] += deposito  # Simetrica
    # Cerrar el ciclo
    T[H[-1]][H[0]] += deposito
    T[H[0]][H[-1]] += deposito
    return T


def Evaporar_Feromonas(T):
    """Evapora feromonas con factor multiplicativo (1-RHO), sin que baje de 0.1"""
    for i in range(len(T)):
        for j in range(len(T[i])):
            T[i][j] = max(T[i][j] * (1 - RHO), 0.1)
    return T


In [ ]:
def hormigas(problem, N, iteraciones=5):
    """Algoritmo de Colonia de Hormigas (ACO)
    problem = datos del problema
    N = Numero de agentes(hormigas) por iteracion
    iteraciones = Numero de rondas completas de hormigas
    """

    # Nodos
    Nodos = list(problem.get_nodes())

    # Inicializa las aristas con una cantidad inicial de feromonas
    T = [[1 for _ in range(len(Nodos))] for _ in range(len(Nodos))]

    mejor_solucion_global = []
    mejor_distancia_global = float('inf')

    for it in range(iteraciones):
        # Se generan los agentes(hormigas) que seran estructuras de caminos desde 0
        Hormiga = [[0] for _ in range(N)]

        # Recorre cada agente construyendo la solucion
        for h in range(N):
            # Para cada agente se construye un camino
            for i in range(len(Nodos)-1):
                Nuevo_Nodo = Add_Nodo(problem, Hormiga[h], T)
                Hormiga[h].append(Nuevo_Nodo)

            # Incrementa feromonas en las aristas recorridas
            T = Incrementa_Feromona(problem, T, Hormiga[h])

        # Evapora Feromonas (una vez por iteracion)
        T = Evaporar_Feromonas(T)

        # Seleccionamos el mejor agente de esta iteracion
        for h in range(N):
            distancia_actual = distancia_total(Hormiga[h], problem)
            if distancia_actual < mejor_distancia_global:
                mejor_solucion_global = Hormiga[h]
                mejor_distancia_global = distancia_actual

        print(f"Iteracion {it+1}/{iteraciones} - Mejor distancia: {mejor_distancia_global}")

    print("
Mejor solucion encontrada:")
    print(mejor_solucion_global)
    print(f"Distancia total: {mejor_distancia_global}")
    return mejor_solucion_global


# Ejecutar ACO con 200 hormigas y 10 iteraciones
sol_aco = hormigas(problem, 200, iteraciones=10)
